# SVCM — ITI-96 — Récupération CodeSystem

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-96
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12207&cid=48534

Récupérer le CodeSystem `TRE-R02-SecteurActivite` par URL canonique, puis par `fullUrl`.

## Configuration

In [1]:
import requests
import json
import time
import os, getpass
from datetime import datetime
from urllib.parse import quote

FHIR_BASE = "https://gazelle-proxypatans.kereval.cloud:10102/fhir"
#FHIR_BASE = os.environ.get("FHIR_BASE", "https://smt.esante.gouv.fr/fhir")

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}
HEADERS_API = {
    "accept": "*/*",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

# ── run output directory ──────────────────────────────────────────────────────
NOTEBOOK_ID = "03_iti96_codesystem"
RUN_TS  = datetime.now().strftime("%Y%m%dT%H%M%S")
RUN_DIR = os.path.join("runs", f"{NOTEBOOK_ID}_{RUN_TS}")
os.makedirs(RUN_DIR, exist_ok=True)

# ── helpers ───────────────────────────────────────────────────────────────────

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def save_artifact(step, filename, data, binary=False):
    """Save a test artifact to the run directory, prefixed with the step number."""
    path = os.path.join(RUN_DIR, f"step{step:03d}_{filename}")
    if binary:
        with open(path, "wb") as f:
            f.write(data)
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✓ {path}")
    return path

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print(f"Configuration OK — sortie dans : {RUN_DIR}")


Configuration OK — sortie dans : runs/03_iti96_codesystem_20260311T211049


---
## Étapes 0–30 — Récupération du CodeSystem TRE-R02-SecteurActivite

**Requête** : `GET /fhir/CodeSystem?url=https://mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite&_format=json`

In [2]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : GET CodeSystem par URL canonique
TRE_R02_URL = "https://mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite"
url_search = f"{FHIR_BASE}/CodeSystem?url={quote(TRE_R02_URL)}&_format=json"

r_search = http_get(url_search)
bundle = r_search.json()

# Étape 20 — Réception et intégration
entries = bundle.get("entry", [])
assert len(entries) > 0, f"Aucune entrée trouvée pour {TRE_R02_URL}"

full_url   = entries[0].get("fullUrl", "")
cs_summary = entries[0].get("resource", {})

print(f"Entrées trouvées : {len(entries)}")
print(f"fullUrl          : {full_url}")
print(f"  name    : {cs_summary.get('name')}")
print(f"  version : {cs_summary.get('version')}")
print(f"  content : {cs_summary.get('content')}")

# Étape 30 — PREUVE : GET par fullUrl
print(f"\n[PREUVE étape 30] GET par fullUrl : {full_url}")
r_direct = http_get(f"{full_url}?_format=json")
cs = r_direct.json()
save_artifact(30, "tre-r02-secteuractivite.json", cs)

print_keys(cs, "resourceType", "id", "url", "version", "name", "title", "status", "content")
concepts = cs.get("concept", [])
print(f"\nNombre de concepts : {len(concepts)}")
if concepts:
    print("Premiers codes :")
    for c in concepts[:10]:
        print(f"  {c.get('code')} — {c.get('display')}")
    if len(concepts) > 10:
        print(f"  ... ({len(concepts) - 10} autres)")


  → GET https://gazelle-proxypatans.kereval.cloud:10102/fhir/CodeSystem?url=https%3A//mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite&_format=json
Entrées trouvées : 1
fullUrl          : https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R02-SecteurActivite
  name    : TRE_R02_SecteurActivite
  version : 20250523120000
  content : complete

[PREUVE étape 30] GET par fullUrl : https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R02-SecteurActivite
  → GET https://smt.esante.gouv.fr/fhir/CodeSystem/TRE-R02-SecteurActivite?_format=json
  ✓ runs/03_iti96_codesystem_20260311T211049/step030_tre-r02-secteuractivite.json
{
  "resourceType": "CodeSystem",
  "id": "TRE-R02-SecteurActivite",
  "url": "https://mos.esante.gouv.fr/NOS/TRE_R02-SecteurActivite/FHIR/TRE-R02-SecteurActivite",
  "version": "20250523120000",
  "name": "TRE_R02_SecteurActivite",
  "status": "active",
  "content": "complete"
}

Nombre de concepts : 69
Premiers codes :
  SA01 — Etablissement public de sant

---
## Récapitulatif

In [3]:
print(f"Run : {RUN_DIR}")
print(f"{'Fichier':<45} {'Taille':>10}")
print("-" * 57)
for fname in sorted(os.listdir(RUN_DIR)):
    fpath = os.path.join(RUN_DIR, fname)
    size  = os.path.getsize(fpath)
    size_str = f"{size/1024:.1f} KB" if size < 1_000_000 else f"{size/1024/1024:.1f} MB"
    print(f"  {fname:<43} {size_str:>10}")
print(f"\n✓ {NOTEBOOK_ID} — terminé.")


Run : runs/03_iti96_codesystem_20260311T211049
Fichier                                           Taille
---------------------------------------------------------
  step030_tre-r02-secteuractivite.json           44.4 KB

✓ 03_iti96_codesystem — terminé.
